# Notebook 05 — SCV-IDS paper-aligned baseline (Step 7)

**Reference:** Chen et al., *SCV-IDS: CNN-ViT Crossattention on Serialized Traffic Images for Spatiotemporal Intrusion Detection*, IEEE IoT Journal 2026, DOI 10.1109/JIOT.2026.3695912.

### Study gate

This experiment fits and evaluates ONLY within the **designated official UNSW-NB15 training partition**. It does not open the official test CSV, predict test samples, or claim the base paper has been outperformed.

**Kaggle dataset attached:** `dsshayan/unsw-nb15` (training file `UNSW_NB15_training-set.csv`). All source files are SHA-256 verified. The predeclared group-disjoint 80/20 training-validation separation ensures identical 42-feature vectors cannot cross the fold boundary.

**Progress bars:** RGB window serialization, per-epoch batches, validation, feature extraction and GitHub uploads.

**Configuration from the paper:** 16×16 24-bit RGB, 5-row windows, 8×8 ViT patches, 4 transformer blocks, 8 heads, width 128, CNN 32/64/128, bidirectional crossattention, AdamW, focal loss, 50 epochs, batch 64, LR=0.001, weight decay=0.001, dropout=0.2.

**Differences from authors' source code:** No official training code is available in this project. The paper omits some quantization and fusion widths, so those choices are explicitly logged. CSV adjacent rows have unverified chronological continuity.

**Reproducibility:** training history, split indices, feature vocabularies, scaler parameters, validation probabilities, t-SNE source vectors, best/last/periodic checkpoints, 28 figure recipes and three figure formats (PNG/SVG/PDF). Large checkpoint and per-row files stay in persistent Kaggle output. GitHub receives the smaller reports, figures, plotting CSVs and SHA-256 manifest.

**CRITICAL:** In Kaggle choose **GPU accelerator**, attach `dsshayan/unsw-nb15`, enable Internet, give access to the existing `pushRare` GitHub-token secret. Do not paste the token into any cell. Once training finishes use **Save Version → Save & Run All** to preserve all the output files.

In [ ]:
from pathlib import Path
import os, subprocess, sys, json

REPO = Path("/kaggle/working/rare-attack-ids-step07")
URL = "https://github.com/itsCodeBakery/rare-attack-spatiotemporal-ids.git"
TRAIN = Path("/kaggle/input/datasets/dsshayan/unsw-nb15/UNSW_NB15_training-set.csv")
assert TRAIN.is_file(), "Attach dsshayan/unsw-nb15 after upload finishes."
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", URL, str(REPO)], check=True)
sha = subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip()
print("RESEARCH GIT COMMIT:",sha)
subprocess.run(["nvidia-smi","--query-gpu=name,memory.total",
                "--format=csv,noheader"],check=True)

In [ ]:
# Kaggle normally preinstalls these dependencies; confirm versions.
modules = ["torch", "numpy", "pandas", "sklearn", "matplotlib", "tqdm"]
for module in modules:
    loaded = __import__(module)
    print(module, getattr(loaded, "__version__", "installed"))
import torch
assert torch.cuda.is_available(), "Please enable GPU in Kaggle Notebook Settings."

In [ ]:
CONFIG=REPO/"configs"/"step07_scv_ids_repro_001.json"
OUTPUT=Path("/kaggle/working/research_outputs/SCV-IDS-REPRO-001")
CHECKPOINTS=Path("/kaggle/working/research_outputs/checkpoints")
environment=os.environ.copy()
environment["PYTHONPATH"]=str(REPO/"src")
# All 50 epochs + checkpoint saves, predictions, 28 figures.
subprocess.run([
    sys.executable, str(REPO/"scripts"/"run_scv_repro.py"),
    "--repo",str(REPO),
    "--config",str(CONFIG),
    "--output",str(OUTPUT),
    "--checkpoints",str(CHECKPOINTS)],
    cwd=str(REPO),env=environment,check=True)
print((OUTPUT/"run_report.md").read_text(encoding="utf-8"))

### Figure reproduction, without retraining

All 28 figures can be rebuilt at any time from the CSVs in `OUTPUT/plot_data`. To regenerate, change the plotting code in GitHub and rerun **only the figure cell below**. The figure manifest records input hashes, captions, and output hashes for every PNG, SVG, and PDF.

The saved `validation_predictions.npz`, `embedding_vectors.npz` and checkpoints are independent reanalysis artifacts and are **not** copied to the public repository; save a Kaggle version so they persist.

In [ ]:
# Optional figure-only rebuild — no model inference and no training.
subprocess.run([
    sys.executable,str(REPO/"scripts"/"plot_scv_figures.py"),
    "--run",str(OUTPUT)],cwd=str(REPO),check=True)
manifest=json.loads((OUTPUT/"figure_manifest.json").read_text())
print("FIGURES:",manifest["figure_count"],
      "FORMATS:",", ".join(manifest["formats"]))

In [ ]:
# Secure GitHub sync of small evidence + all PNG/SVG/PDF figures.
try:
    from kaggle_secrets import UserSecretsClient
    token=UserSecretsClient().get_secret("pushRare")
except Exception:
    token=None
if not token:
    print("Not synchronized: give notebook access to Kaggle secret pushRare.")
else:
    env=os.environ.copy()
    env["GITHUB_TOKEN"]=token
    try:
        subprocess.run([
            sys.executable,str(REPO/"scripts"/"sync_scv_results.py"),
            "--run",str(OUTPUT)],cwd=str(REPO),env=env,check=True)
    finally:
        env.pop("GITHUB_TOKEN",None)
        del token

## Research data retention

Before closing Kaggle, **Save Version → Save & Run All** (or another Kaggle workflow that preserves all `/kaggle/working/research_outputs` files). For an interrupted run, use the same output folder and add `--resume` to the training command; note Kaggle's working directory is not guaranteed to survive a new session. Preserve snapshots in the saved Kaggle run.

Send the final `VERIFIED SCV BASELINE SYNC COMMIT:` URL. We will then verify figure files, compute metric consistency, and decide whether to replicate more seeds or revise the baseline. **Do not open or train using the official UNSW test file in this notebook.**